In [9]:
import pandas as pd
import numpy as np
import tpqoa
import Structures
import DatasetStructure as ds
from DatasetStructure import DatasetStructure
from Structures import Structures

In [2]:
api = tpqoa.tpqoa("oanda.cfg")

In [79]:
api.get_instruments()

[('AUD/CAD', 'AUD_CAD'),
 ('AUD/CHF', 'AUD_CHF'),
 ('AUD/HKD', 'AUD_HKD'),
 ('AUD/JPY', 'AUD_JPY'),
 ('AUD/NZD', 'AUD_NZD'),
 ('AUD/SGD', 'AUD_SGD'),
 ('AUD/USD', 'AUD_USD'),
 ('Australia 200', 'AU200_AUD'),
 ('Bitcoin', 'BTC_USD'),
 ('Bitcoin Cash', 'BCH_USD'),
 ('Brent Crude Oil', 'BCO_USD'),
 ('Bund', 'DE10YB_EUR'),
 ('CAD/CHF', 'CAD_CHF'),
 ('CAD/HKD', 'CAD_HKD'),
 ('CAD/JPY', 'CAD_JPY'),
 ('CAD/SGD', 'CAD_SGD'),
 ('CHF/HKD', 'CHF_HKD'),
 ('CHF/JPY', 'CHF_JPY'),
 ('CHF/ZAR', 'CHF_ZAR'),
 ('China A50', 'CN50_USD'),
 ('China H Shares', 'CHINAH_HKD'),
 ('Copper', 'XCU_USD'),
 ('Corn', 'CORN_USD'),
 ('EUR/AUD', 'EUR_AUD'),
 ('EUR/CAD', 'EUR_CAD'),
 ('EUR/CHF', 'EUR_CHF'),
 ('EUR/CZK', 'EUR_CZK'),
 ('EUR/DKK', 'EUR_DKK'),
 ('EUR/GBP', 'EUR_GBP'),
 ('EUR/HKD', 'EUR_HKD'),
 ('EUR/HUF', 'EUR_HUF'),
 ('EUR/JPY', 'EUR_JPY'),
 ('EUR/NOK', 'EUR_NOK'),
 ('EUR/NZD', 'EUR_NZD'),
 ('EUR/PLN', 'EUR_PLN'),
 ('EUR/SEK', 'EUR_SEK'),
 ('EUR/SGD', 'EUR_SGD'),
 ('EUR/TRY', 'EUR_TRY'),
 ('EUR/USD', 'EUR_U

In [82]:
df = api.get_history('EUR_USD', "2025-01-01", "2026-02-16", "H1", "B")
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6962 entries, 2025-01-01 22:00:00 to 2026-02-15 23:00:00
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   o         6962 non-null   float64
 1   h         6962 non-null   float64
 2   l         6962 non-null   float64
 3   c         6962 non-null   float64
 4   volume    6962 non-null   int64  
 5   complete  6962 non-null   bool   
dtypes: bool(1), float64(4), int64(1)
memory usage: 333.1 KB


In [83]:
df.to_csv("eurusd.csv")

In [84]:
data = DatasetStructure('eurusd.csv')

In [85]:
data.ATR(14)

In [86]:
data = data.show_data()

In [88]:
data['impulse_candle'] = np.where((data['body'] >= 1.7 * data['ATR_14']) &
                                          (data['body'] / data['total_range'] > 0.7)
                                          , True, None)

In [94]:
data['impulse_candle'].dropna().tail(10)

time
2026-01-20 08:00:00    True
2026-01-21 12:00:00    True
2026-01-23 17:00:00    True
2026-01-27 09:00:00    True
2026-01-27 12:00:00    True
2026-02-03 09:00:00    True
2026-02-05 09:00:00    True
2026-02-06 13:00:00    True
2026-02-09 06:00:00    True
2026-02-09 13:00:00    True
Name: impulse_candle, dtype: object

In [90]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6962 entries, 2025-01-01 22:00:00 to 2026-02-15 23:00:00
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   o               6962 non-null   float64
 1   h               6962 non-null   float64
 2   l               6962 non-null   float64
 3   c               6962 non-null   float64
 4   volume          6962 non-null   int64  
 5   complete        6962 non-null   bool   
 6   body            6962 non-null   float64
 7   upper_wick      6962 non-null   float64
 8   lower_wick      6962 non-null   float64
 9   total_range     6962 non-null   float64
 10  true_range      6961 non-null   float64
 11  direction       6962 non-null   float64
 12  ATR_14          6948 non-null   float64
 13  impulse_candle  148 non-null    object 
dtypes: bool(1), float64(11), int64(1), object(1)
memory usage: 768.3+ KB


In [99]:
data['order_block'] = np.where((data['impulse_candle'].shift(-1))&
                               (data['direction'] != data["direction"].shift(-1))&
                               (data['body']<= 0.33 * data['body'].shift(-1)),
                               True, None)

In [100]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6962 entries, 2025-01-01 22:00:00 to 2026-02-15 23:00:00
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   o               6962 non-null   float64
 1   h               6962 non-null   float64
 2   l               6962 non-null   float64
 3   c               6962 non-null   float64
 4   volume          6962 non-null   int64  
 5   complete        6962 non-null   bool   
 6   body            6962 non-null   float64
 7   upper_wick      6962 non-null   float64
 8   lower_wick      6962 non-null   float64
 9   total_range     6962 non-null   float64
 10  true_range      6961 non-null   float64
 11  direction       6962 non-null   float64
 12  ATR_14          6948 non-null   float64
 13  impulse_candle  148 non-null    object 
 14  order_block     54 non-null     object 
dtypes: bool(1), float64(11), int64(1), object(2)
memory usage: 822.7+ KB


In [104]:
data['order_block'].to_csv('orderblock.csv')